# Milestone 17 Objective

Build the standalone UI Testing Agent before integrating it into the main LangGraph workflow.

## UI Testing Agent Role

The UI Agent reads planned UI tests from `test_plan`, opens `target_url` with Selenium, checks stable page/form/text evidence, and saves `ui_result.json` plus screenshots on failure.

## Why It Works Alone First

The agent is tested in isolation so browser availability, target availability, screenshots, and result schemas can be verified before changing the full workflow.

## Mini LangGraph Workflow

`START -> ui_testing -> END`

## State Read

The UI Agent reads `target_url`, `test_plan`, `user_preferences`, and `discovered_ui_flows`.

## State Write

The UI Agent writes `ui_results`, `ui_result_path`, `screenshots`, `errors`, and `agent_logs`.

## Safety Note

The agent does not start the target app. If the app or browser is unavailable, it records `environment_error` and does not crash.

## Target Repository

https://github.com/Vitaee/DjangoRestAPI

## Target URL

http://localhost:8000

In [1]:
# Part A - unit-style execution with mocked Selenium
from pathlib import Path
from test_auto.agents import ui_testing_agent

fake_test_plan = {
    "ui_tests": [
        {"id": "UI_001", "name": "login_page_visible", "flow": "login", "expected_result": "login form is visible"},
        {"id": "UI_002", "name": "register_page_visible", "flow": "register", "expected_result": "register form is visible"},
        {"id": "UI_003", "name": "todo_home_visible", "flow": "todo", "expected_result": "todo home is visible"},
    ]
}

def fake_execute_ui_test_case(target_url, test_case, run_id, discovered_ui_flows=None, user_preferences=None):
    return {
        "id": test_case["id"],
        "name": test_case["name"],
        "flow": test_case.get("flow"),
        "status": "passed",
        "target_path": "/",
        "target_url": target_url,
        "duration_ms": 5.0,
        "details": "mocked Selenium execution",
        "screenshot": None,
        "assertions": [{"type": "ui_visible", "passed": True}],
        "error_type": None,
        "evidence": {"title": "Mock"},
    }

original_execute = ui_testing_agent.execute_ui_test_case
ui_testing_agent.execute_ui_test_case = fake_execute_ui_test_case
try:
    result = ui_testing_agent.run_ui_testing_agent_alone(
        target_url="http://localhost:8000",
        test_plan=fake_test_plan,
        run_id="ui_notebook_mocked",
    )
finally:
    ui_testing_agent.execute_ui_test_case = original_execute

{
    "summary": result["summary"],
    "ui_result_path": result["ui_result_path"],
    "ui_results": result["ui_results"],
}

{'summary': {'total_tests': 3,
  'passed': 3,
  'failed': 0,
  'skipped': 0,
  'errors': 0,
  'pass_rate': 100.0},
 'ui_result_path': 'results\\runs\\ui_notebook_mocked\\ui_result.json',
 'ui_results': {'agent': 'ui_testing',
  'timestamp': '2026-05-20T08:42:48.246265+00:00',
  'status': 'success',
  'duration_seconds': 4.7600013203918934e-05,
  'summary': {'total_tests': 3,
   'passed': 3,
   'failed': 0,
   'skipped': 0,
   'errors': 0,
   'pass_rate': 100.0},
  'tests': [{'id': 'UI_001',
    'name': 'login_page_visible',
    'flow': 'login',
    'status': 'passed',
    'target_path': '/',
    'target_url': 'http://localhost:8000',
    'duration_ms': 5.0,
    'details': 'mocked Selenium execution',
    'screenshot': None,
    'assertions': [{'type': 'ui_visible', 'passed': True}],
    'error_type': None,
    'evidence': {'title': 'Mock'}},
   {'id': 'UI_002',
    'name': 'register_page_visible',
    'flow': 'register',
    'status': 'passed',
    'target_path': '/',
    'target_url':

## Part B - Optional Real Target App

This requires the Django target app running at http://localhost:8000 and a browser available. Do not start the Django app from this notebook.

In [2]:
# Optional real run from a previous workflow run directory
# from pathlib import Path
# from test_auto.agents.ui_testing_agent import load_test_plan_from_run_dir, run_ui_testing_agent_alone
# run_dir = Path("results/runs/<run_id>")
# test_plan = load_test_plan_from_run_dir(run_dir)
# real_result = run_ui_testing_agent_alone(target_url="http://localhost:8000", test_plan=test_plan)
# {
#     "summary": real_result["summary"],
#     "screenshots": real_result["screenshots"],
#     "ui_result_path": real_result["ui_result_path"],
# }


In [3]:
# Part C - mini LangGraph workflow
from test_auto.graph.ui_testing_workflow import run_ui_testing_workflow
from test_auto.agents import ui_testing_agent

original_execute = ui_testing_agent.execute_ui_test_case
ui_testing_agent.execute_ui_test_case = fake_execute_ui_test_case
try:
    final_state = run_ui_testing_workflow(
        {
            "run_id": "ui_workflow_notebook_mocked",
            "target_url": "http://localhost:8000",
            "test_plan": fake_test_plan,
            "errors": [],
            "agent_logs": [],
        }
    )
finally:
    ui_testing_agent.execute_ui_test_case = original_execute

final_state

{'run_id': 'ui_workflow_notebook_mocked',
 'target_url': 'http://localhost:8000',
 'errors': [],
 'agent_logs': [{'agent': 'ui_testing',
   'timestamp': '2026-05-20T08:42:48.796520+00:00',
   'status': 'success',
   'duration_seconds': 4.549999721348286e-05,
   'summary': {'total_tests': 3,
    'passed': 3,
    'failed': 0,
    'skipped': 0,
    'errors': 0,
    'pass_rate': 100.0},
   'tests': [{'id': 'UI_001',
     'name': 'login_page_visible',
     'flow': 'login',
     'status': 'passed',
     'target_path': '/',
     'target_url': 'http://localhost:8000',
     'duration_ms': 5.0,
     'details': 'mocked Selenium execution',
     'screenshot': None,
     'assertions': [{'type': 'ui_visible', 'passed': True}],
     'error_type': None,
     'evidence': {'title': 'Mock'}},
    {'id': 'UI_002',
     'name': 'register_page_visible',
     'flow': 'register',
     'status': 'passed',
     'target_path': '/',
     'target_url': 'http://localhost:8000',
     'duration_ms': 5.0,
     'detail

In [4]:
# Part D - graph visualization
from test_auto.graph.ui_testing_workflow import build_ui_testing_graph

graph = build_ui_testing_graph()
try:
    png = graph.get_graph().draw_mermaid_png()
    print(f"Rendered graph PNG bytes: {len(png)}")
except Exception:
    try:
        print(graph.get_graph().draw_mermaid())
    except Exception:
        print("START -> ui_testing -> END")


Rendered graph PNG bytes: 6288
